# Google Play Controlled Scale Ingestion — Phase 2 Day 2

This notebook runs Phase 2 Day 2 of the Google Play review ingestion pipeline.

Phase 2 Day 1 created the first larger controlled baseline database with 12,000 Google Play reviews across 10 apps.

Phase 2 Day 2 uses the clean Day 1 database and repeats the same ingestion scope to test:

- duplicate handling
- new review capture
- repeated ingestion stability
- app-level errors
- quality flag patterns
- database growth
- Day 1 vs Day 2 comparison

Important:

This notebook must start from the clean Phase 2 Day 1 database.  
The clean Day 1 database should contain:

- exactly 1 Phase 2 Day 1 run
- 12,000 raw reviews
- 12,000 cleaned reviews
- 10 app-level summary rows
- no failed Day 1 run
- no previous Day 2 run

In [1]:
from pathlib import Path
import os

REPO_URL = "https://github.com/Yaxuanzhang5/app-review-source-validation.git"
REPO_DIR = Path("/content/app-review-source-validation")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo folder already exists.")

%cd {REPO_DIR}

Cloning into '/content/app-review-source-validation'...
remote: Enumerating objects: 292, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 292 (delta 3), reused 0 (delta 0), pack-reused 272 (from 1)
Receiving objects: 100% (292/292), 8.48 MiB | 9.86 MiB/s, done.
Resolving deltas: 100% (126/126), done.
/content/app-review-source-validation


In [2]:
!pip install -q google-play-scraper pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.3 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import time
import html
import shutil
import sqlite3
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from google_play_scraper import app, reviews, Sort
from google.colab import files

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

print("Packages imported.")

Packages imported.


## 1. Day 2 Configuration

This run keeps the same app list and same review target as Phase 2 Day 1.

The purpose is to make Day 1 and Day 2 directly comparable.

In [4]:
PHASE = "phase2"
RUN_LABEL = "phase2_day2_daily_followup"
FREQUENCY_LABEL = "daily_followup"

SOURCE = "google_play"
LANGUAGE = "en"
COUNTRY = "us"
TARGET_REVIEWS_PER_APP = 1200

DB_PATH = "database/google_play_reviews.sqlite"

OUTPUT_DIR = Path("outputs")
RUN_SUMMARY_DIR = OUTPUT_DIR / "run_summaries"
QUALITY_DIR = OUTPUT_DIR / "quality"
REPORT_DIR = Path("reports")
BACKUP_DIR = Path("database") / "backups"

for folder in [RUN_SUMMARY_DIR, QUALITY_DIR, REPORT_DIR, BACKUP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

APPS = {
    "YouTube": "com.google.android.youtube",
    "TikTok": "com.zhiliaoapp.musically",
    "Spotify": "com.spotify.music",
    "Instagram": "com.instagram.android",
    "Uber": "com.ubercab",
    "DoorDash": "com.dd.doordash",
    "Duolingo": "com.duolingo",
    "Google Maps": "com.google.android.apps.maps",
    "Netflix": "com.netflix.mediaclient",
    "Reddit": "com.reddit.frontpage"
}

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ID = f"{RUN_LABEL}_{run_timestamp}"

print("Run ID:", RUN_ID)
print("Database path:", DB_PATH)
print("App count:", len(APPS))
print("Target reviews per app:", TARGET_REVIEWS_PER_APP)
print("Frequency label:", FREQUENCY_LABEL)

Run ID: phase2_day2_daily_followup_20260708_041135
Database path: database/google_play_reviews.sqlite
App count: 10
Target reviews per app: 1200
Frequency label: daily_followup


## 2. Upload Clean Phase 2 Day 1 Zip

Upload the clean Day 1 zip file only.

The expected file is the clean Day 1 package that contains:

`database/google_play_reviews.sqlite`

Do not upload an old single `.sqlite` file here.

In [5]:
uploaded = files.upload()

if len(uploaded) == 0:
    raise FileNotFoundError("No file uploaded. Please upload the clean Phase 2 Day 1 zip file.")

db_target_path = Path(DB_PATH)
db_target_path.parent.mkdir(parents=True, exist_ok=True)

uploaded_names = list(uploaded.keys())

if len(uploaded_names) != 1:
    raise ValueError("Please upload exactly one clean Day 1 zip file.")

uploaded_name = uploaded_names[0]

possible_paths = [
    Path.cwd() / uploaded_name,
    Path("/content") / uploaded_name
]

uploaded_path = None

for path in possible_paths:
    if path.exists():
        uploaded_path = path
        break

if uploaded_path is None:
    raise FileNotFoundError(f"Uploaded file was not found: {uploaded_name}")

print("Uploaded file:", uploaded_name)
print("Found uploaded file at:", uploaded_path)

if not uploaded_name.endswith(".zip"):
    raise ValueError(
        "Please upload the clean Day 1 zip file, not a single sqlite file. "
        "This prevents accidentally using an old or dirty database."
    )

with zipfile.ZipFile(uploaded_path, "r") as zip_ref:
    zip_names = zip_ref.namelist()

    db_candidates = [
        name for name in zip_names
        if name.endswith("database/google_play_reviews.sqlite")
        or name.endswith("google_play_reviews.sqlite")
    ]

    if len(db_candidates) == 0:
        raise FileNotFoundError("No google_play_reviews.sqlite found inside the uploaded zip file.")

    selected_db_in_zip = db_candidates[0]
    print("Database found in zip:", selected_db_in_zip)

    extracted_path = Path.cwd() / "uploaded_clean_day1_database.sqlite"

    with zip_ref.open(selected_db_in_zip) as source_file:
        with open(extracted_path, "wb") as target_file:
            shutil.copyfileobj(source_file, target_file)

shutil.copy2(extracted_path, db_target_path)

if not db_target_path.exists():
    raise FileNotFoundError("Database was not copied successfully.")

db_size_mb = os.path.getsize(db_target_path) / (1024 * 1024)

print("Database copied to:", db_target_path)
print(f"Database size after upload: {db_size_mb:.4f} MB")

if db_size_mb < 20:
    raise ValueError(
        "This database is smaller than expected. "
        "The clean Day 1 database should be around 25 MB after inserting 12,000 reviews."
    )

print("Clean Day 1 database file loaded.")

Saving google_play_reviews.sqlite.zip to google_play_reviews.sqlite.zip
Uploaded file: google_play_reviews.sqlite.zip
Found uploaded file at: /content/app-review-source-validation/google_play_reviews.sqlite.zip
Database found in zip: google_play_reviews.sqlite
Database copied to: database/google_play_reviews.sqlite
Database size after upload: 25.4922 MB
Clean Day 1 database file loaded.


## 3. Connect to Database and Define Helper Functions

In [6]:
def get_db_size_mb(path):
    if os.path.exists(path):
        return os.path.getsize(path) / (1024 * 1024)
    return 0

def get_tables(conn):
    query = """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """
    return pd.read_sql_query(query, conn)

def table_exists(conn, table_name):
    query = """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table' AND name = ?;
    """
    result = pd.read_sql_query(query, conn, params=[table_name])
    return len(result) > 0

def count_rows(conn, table_name):
    if not table_exists(conn, table_name):
        return 0

    query = f"SELECT COUNT(*) AS row_count FROM {table_name};"
    return int(pd.read_sql_query(query, conn)["row_count"].iloc[0])

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

print("Database connection opened.")
print("Database size:", round(get_db_size_mb(DB_PATH), 4), "MB")

Database connection opened.
Database size: 25.4922 MB


## 4. Make Sure Phase 2 Tables Exist

This should not reset or delete anything.  
It only ensures the required tables are available.

In [7]:
create_schema_sql = """
CREATE TABLE IF NOT EXISTS phase2_ingestion_runs (
    run_id TEXT PRIMARY KEY,
    run_label TEXT,
    phase TEXT,
    frequency_label TEXT,
    source TEXT,
    language TEXT,
    country TEXT,
    target_reviews_per_app INTEGER,
    app_count INTEGER,
    apps_included TEXT,
    run_started_at TEXT,
    run_finished_at TEXT,
    runtime_seconds REAL,
    status TEXT,
    records_fetched_total INTEGER DEFAULT 0,
    new_records_inserted_total INTEGER DEFAULT 0,
    duplicates_skipped_total INTEGER DEFAULT 0,
    errors_total INTEGER DEFAULT 0,
    apps_failed TEXT,
    quality_flag_total INTEGER DEFAULT 0,
    quality_flags_inserted INTEGER DEFAULT 0,
    db_size_before_mb REAL,
    db_size_after_mb REAL,
    db_size_growth_mb REAL,
    review_rows_before INTEGER,
    review_rows_after INTEGER,
    review_rows_growth INTEGER,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS phase2_apps (
    app_id TEXT PRIMARY KEY,
    app_name TEXT,
    source TEXT,
    language TEXT,
    country TEXT,
    title_from_store TEXT,
    score_from_store REAL,
    ratings_from_store INTEGER,
    installs_from_store TEXT,
    last_validated_at TEXT,
    last_validation_status TEXT,
    last_validation_error TEXT
);

CREATE TABLE IF NOT EXISTS phase2_reviews_raw (
    review_key TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    app_id TEXT NOT NULL,
    app_name TEXT,
    review_id TEXT,
    user_name TEXT,
    user_image TEXT,
    content_raw TEXT,
    score INTEGER,
    thumbs_up_count INTEGER,
    review_created_at TEXT,
    reply_content_raw TEXT,
    replied_at TEXT,
    app_version TEXT,
    fetched_at TEXT,
    run_id TEXT,
    raw_json TEXT,
    UNIQUE(source, app_id, review_id),
    FOREIGN KEY(app_id) REFERENCES phase2_apps(app_id),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);

CREATE TABLE IF NOT EXISTS phase2_reviews_cleaned (
    review_key TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    app_id TEXT NOT NULL,
    content_cleaned TEXT,
    content_length INTEGER,
    has_developer_reply INTEGER,
    score INTEGER,
    review_created_at TEXT,
    app_version TEXT,
    cleaned_at TEXT,
    run_id TEXT,
    FOREIGN KEY(review_key) REFERENCES phase2_reviews_raw(review_key),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);

CREATE TABLE IF NOT EXISTS phase2_quality_flags (
    flag_id TEXT PRIMARY KEY,
    review_key TEXT,
    run_id TEXT,
    app_id TEXT,
    flag_name TEXT,
    flag_severity TEXT,
    flag_value TEXT,
    created_at TEXT,
    FOREIGN KEY(review_key) REFERENCES phase2_reviews_raw(review_key),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);

CREATE TABLE IF NOT EXISTS phase2_app_run_summary (
    run_id TEXT,
    app_name TEXT,
    app_id TEXT,
    target_reviews INTEGER,
    records_fetched INTEGER,
    unique_reviews_in_batch INTEGER,
    duplicate_reviews_in_batch INTEGER,
    new_records_inserted INTEGER,
    duplicates_skipped INTEGER,
    runtime_seconds REAL,
    min_review_date TEXT,
    max_review_date TEXT,
    missing_review_id_count INTEGER,
    missing_content_count INTEGER,
    empty_content_count INTEGER,
    missing_score_count INTEGER,
    invalid_score_count INTEGER,
    missing_review_date_count INTEGER,
    missing_app_version_count INTEGER,
    missing_developer_reply_count INTEGER,
    quality_flag_count INTEGER,
    error_message TEXT,
    PRIMARY KEY(run_id, app_id),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);
"""

conn.executescript(create_schema_sql)
conn.commit()

print("Phase 2 tables are ready.")
display(get_tables(conn))

Phase 2 tables are ready.


,name
0,app_sources
1,ingestion_run_targets
2,ingestion_runs
3,phase2_app_run_summary
4,phase2_apps
5,phase2_ingestion_runs
6,phase2_quality_flags
7,phase2_reviews_cleaned
8,phase2_reviews_raw
9,review_quality_flags


## 5. Inspect Database Before Day 2

In [8]:
tables_before_df = get_tables(conn)

row_counts_before = []

for table_name in tables_before_df["name"]:
    row_counts_before.append({
        "table_name": table_name,
        "row_count_before": count_rows(conn, table_name)
    })

row_counts_before_df = pd.DataFrame(row_counts_before)

print("Row counts before Phase 2 Day 2:")
display(row_counts_before_df)

Row counts before Phase 2 Day 2:


,table_name,row_count_before
0,app_sources,3
1,ingestion_run_targets,12
2,ingestion_runs,4
3,phase2_app_run_summary,20
4,phase2_apps,10
5,phase2_ingestion_runs,2
6,phase2_quality_flags,12633
7,phase2_reviews_cleaned,12000
8,phase2_reviews_raw,12000
9,review_quality_flags,1200


## 6. Strict Safety Check Before Day 2

This is the most important check in this notebook.

The database must be a clean Day 1 database:

- exactly one Day 1 run
- Day 1 status must be completed
- Day 1 inserted 12,000 records
- raw review table must contain 12,000 rows
- app-level summary must contain 10 rows
- no failed Day 1 runs
- no existing Day 2 runs

In [10]:
# Clean failed Day 1 run records from the uploaded database
# This only removes failed tracking / summary rows.
# It does not remove the correct 12,000 Day 1 reviews.

failed_day1_to_remove_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        records_fetched_total,
        new_records_inserted_total,
        errors_total,
        review_rows_after
    FROM phase2_ingestion_runs
    WHERE run_label = 'phase2_day1_controlled_scale'
      AND (
          status != 'completed'
          OR new_records_inserted_total != 12000
          OR errors_total != 0
          OR review_rows_after != 12000
      );
    """,
    conn
)

print("Failed / dirty Day 1 runs to remove:")
display(failed_day1_to_remove_df)

if len(failed_day1_to_remove_df) == 0:
    print("No failed Day 1 runs found. Nothing to clean.")
else:
    # Make sure we only remove failed runs that did not insert useful review rows
    unsafe_failed_runs = failed_day1_to_remove_df[
        failed_day1_to_remove_df["new_records_inserted_total"].fillna(0).astype(int) > 0
    ]

    if len(unsafe_failed_runs) > 0:
        display(unsafe_failed_runs)
        raise ValueError(
            "Some failed Day 1 runs inserted rows. Do not clean automatically. "
            "Please review manually."
        )

    failed_run_ids = failed_day1_to_remove_df["run_id"].tolist()

    for failed_run_id in failed_run_ids:
        print("Removing failed Day 1 run:", failed_run_id)

        conn.execute(
            "DELETE FROM phase2_quality_flags WHERE run_id = ?;",
            (failed_run_id,)
        )

        conn.execute(
            "DELETE FROM phase2_app_run_summary WHERE run_id = ?;",
            (failed_run_id,)
        )

        conn.execute(
            "DELETE FROM phase2_reviews_cleaned WHERE run_id = ?;",
            (failed_run_id,)
        )

        conn.execute(
            "DELETE FROM phase2_reviews_raw WHERE run_id = ?;",
            (failed_run_id,)
        )

        conn.execute(
            "DELETE FROM phase2_ingestion_runs WHERE run_id = ?;",
            (failed_run_id,)
        )

    conn.commit()
    print("Failed Day 1 run records removed.")

# Re-check after cleanup
post_clean_run_check_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        review_rows_before,
        review_rows_after
    FROM phase2_ingestion_runs
    ORDER BY run_started_at;
    """,
    conn
)

post_clean_counts_df = pd.DataFrame({
    "table_name": [
        "phase2_ingestion_runs",
        "phase2_app_run_summary",
        "phase2_reviews_raw",
        "phase2_reviews_cleaned",
        "phase2_quality_flags"
    ],
    "row_count": [
        count_rows(conn, "phase2_ingestion_runs"),
        count_rows(conn, "phase2_app_run_summary"),
        count_rows(conn, "phase2_reviews_raw"),
        count_rows(conn, "phase2_reviews_cleaned"),
        count_rows(conn, "phase2_quality_flags")
    ]
})

print("\nPost-clean run check:")
display(post_clean_run_check_df)

print("\nPost-clean table counts:")
display(post_clean_counts_df)

Failed / dirty Day 1 runs to remove:


,run_id,run_label,status,records_fetched_total,new_records_inserted_total,errors_total,review_rows_after
0,phase2_day1_controlled_scale_20260707_035428,phase2_day1_controlled_scale,completed_with_errors,12000,0,10,0


Removing failed Day 1 run: phase2_day1_controlled_scale_20260707_035428
Failed Day 1 run records removed.

Post-clean run check:


,run_id,run_label,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,review_rows_before,review_rows_after
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,completed,12000,12000,0,0,0,12000



Post-clean table counts:


,table_name,row_count
0,phase2_ingestion_runs,1
1,phase2_app_run_summary,10
2,phase2_reviews_raw,12000
3,phase2_reviews_cleaned,12000
4,phase2_quality_flags,12633


In [11]:
all_phase2_runs_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        review_rows_before,
        review_rows_after,
        db_size_before_mb,
        db_size_after_mb
    FROM phase2_ingestion_runs
    ORDER BY run_started_at;
    """,
    conn
)

day1_check_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        review_rows_before,
        review_rows_after,
        db_size_before_mb,
        db_size_after_mb
    FROM phase2_ingestion_runs
    WHERE run_label = 'phase2_day1_controlled_scale'
    ORDER BY run_started_at;
    """,
    conn
)

failed_day1_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        new_records_inserted_total,
        errors_total,
        notes
    FROM phase2_ingestion_runs
    WHERE run_label = 'phase2_day1_controlled_scale'
      AND status != 'completed';
    """,
    conn
)

existing_day2_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        run_started_at
    FROM phase2_ingestion_runs
    WHERE run_label = 'phase2_day2_daily_followup';
    """,
    conn
)

existing_phase2_raw_rows = count_rows(conn, "phase2_reviews_raw")
existing_phase2_cleaned_rows = count_rows(conn, "phase2_reviews_cleaned")
existing_phase2_quality_rows = count_rows(conn, "phase2_quality_flags")
existing_app_summary_rows = count_rows(conn, "phase2_app_run_summary")
current_db_size_mb = get_db_size_mb(DB_PATH)

print("Current database size:", round(current_db_size_mb, 4), "MB")
print("Existing phase2 raw review rows:", existing_phase2_raw_rows)
print("Existing phase2 cleaned review rows:", existing_phase2_cleaned_rows)
print("Existing phase2 quality flag rows:", existing_phase2_quality_rows)
print("Existing app summary rows:", existing_app_summary_rows)

print("\nAll Phase 2 runs currently in database:")
display(all_phase2_runs_df)

print("\nDay 1 run check:")
display(day1_check_df)

print("\nFailed Day 1 runs:")
display(failed_day1_df)

print("\nExisting Day 2 runs:")
display(existing_day2_df)

if len(day1_check_df) != 1:
    raise ValueError(
        f"Expected exactly 1 clean Day 1 run, but found {len(day1_check_df)}. "
        "Please upload the clean Day 1 zip."
    )

day1_row = day1_check_df.iloc[0]

if day1_row["status"] != "completed":
    raise ValueError("Day 1 run status is not completed. Please upload the clean Day 1 zip.")

if int(day1_row["records_fetched_total"]) != 12000:
    raise ValueError("Day 1 records_fetched_total is not 12,000.")

if int(day1_row["new_records_inserted_total"]) != 12000:
    raise ValueError("Day 1 new_records_inserted_total is not 12,000.")

if int(day1_row["duplicates_skipped_total"]) != 0:
    raise ValueError("Day 1 duplicates_skipped_total should be 0 for the baseline run.")

if int(day1_row["errors_total"]) != 0:
    raise ValueError("Day 1 errors_total should be 0.")

if int(day1_row["review_rows_after"]) != 12000:
    raise ValueError("Day 1 review_rows_after should be 12,000.")

if existing_phase2_raw_rows != 12000:
    raise ValueError("phase2_reviews_raw should contain exactly 12,000 rows before Day 2.")

if existing_phase2_cleaned_rows != 12000:
    raise ValueError("phase2_reviews_cleaned should contain exactly 12,000 rows before Day 2.")

if existing_app_summary_rows != 10:
    raise ValueError("phase2_app_run_summary should contain exactly 10 rows before Day 2.")

if len(failed_day1_df) > 0:
    raise ValueError("Failed Day 1 runs found. Please upload the clean Day 1 zip.")

if len(existing_day2_df) > 0:
    raise ValueError("Existing Day 2 runs found. Please start from the clean Day 1 database.")

print("Strict safety check passed. This database is ready for Phase 2 Day 2.")

Current database size: 25.4922 MB
Existing phase2 raw review rows: 12000
Existing phase2 cleaned review rows: 12000
Existing phase2 quality flag rows: 12633
Existing app summary rows: 10

All Phase 2 runs currently in database:


,run_id,run_label,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,review_rows_before,review_rows_after,db_size_before_mb,db_size_after_mb
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,completed,12000,12000,0,0,0,12000,1.832031,25.492188



Day 1 run check:


,run_id,run_label,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,review_rows_before,review_rows_after,db_size_before_mb,db_size_after_mb
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,completed,12000,12000,0,0,0,12000,1.832031,25.492188



Failed Day 1 runs:


,run_id,run_label,status,new_records_inserted_total,errors_total,notes



Existing Day 2 runs:


,run_id,run_label,status,run_started_at


Strict safety check passed. This database is ready for Phase 2 Day 2.


## 7. Create Backup Before Day 2

This backup is only for local safety.  
It does not need to be uploaded to GitHub.

In [12]:
db_size_before_backup_mb = get_db_size_mb(DB_PATH)
backup_path = BACKUP_DIR / f"google_play_reviews_before_{RUN_ID}.sqlite"

shutil.copy2(DB_PATH, backup_path)

print(f"Database exists: {DB_PATH}")
print(f"Current database size before backup: {db_size_before_backup_mb:.4f} MB")
print(f"Backup saved to: {backup_path}")

Database exists: database/google_play_reviews.sqlite
Current database size before backup: 25.4922 MB
Backup saved to: database/backups/google_play_reviews_before_phase2_day2_daily_followup_20260708_041135.sqlite


## 8. Start Day 2 Ingestion Run Record

In [13]:
run_started_at = datetime.now(timezone.utc)
run_started_at_text = run_started_at.isoformat()

review_rows_before = count_rows(conn, "phase2_reviews_raw")
db_size_before_mb = get_db_size_mb(DB_PATH)

conn.execute(
    """
    INSERT INTO phase2_ingestion_runs (
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        status,
        db_size_before_mb,
        review_rows_before,
        notes
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """,
    (
        RUN_ID,
        RUN_LABEL,
        PHASE,
        FREQUENCY_LABEL,
        SOURCE,
        LANGUAGE,
        COUNTRY,
        TARGET_REVIEWS_PER_APP,
        len(APPS),
        ", ".join(APPS.keys()),
        run_started_at_text,
        "started",
        db_size_before_mb,
        review_rows_before,
        "Phase 2 Day 2 daily follow-up ingestion run from clean Day 1 database."
    )
)

conn.commit()

print("Run tracking row created.")
print("Run started at:", run_started_at_text)
print(f"Database size before run: {db_size_before_mb:.4f} MB")
print("Phase 2 raw review rows before run:", review_rows_before)

Run tracking row created.
Run started at: 2026-07-08T04:16:09.351152+00:00
Database size before run: 25.4922 MB
Phase 2 raw review rows before run: 12000


## 9. Validate App IDs

In [14]:
validation_rows = []
validated_at = datetime.now(timezone.utc).isoformat()

for app_name, app_id in APPS.items():
    try:
        store_info = app(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY
        )

        validation_rows.append({
            "app_name": app_name,
            "app_id": app_id,
            "validation_status": "ok",
            "title_from_store": store_info.get("title"),
            "score_from_store": store_info.get("score"),
            "ratings_from_store": store_info.get("ratings"),
            "installs_from_store": store_info.get("installs"),
            "validation_error": ""
        })

        conn.execute(
            """
            INSERT INTO phase2_apps (
                app_id,
                app_name,
                source,
                language,
                country,
                title_from_store,
                score_from_store,
                ratings_from_store,
                installs_from_store,
                last_validated_at,
                last_validation_status,
                last_validation_error
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(app_id) DO UPDATE SET
                app_name = excluded.app_name,
                source = excluded.source,
                language = excluded.language,
                country = excluded.country,
                title_from_store = excluded.title_from_store,
                score_from_store = excluded.score_from_store,
                ratings_from_store = excluded.ratings_from_store,
                installs_from_store = excluded.installs_from_store,
                last_validated_at = excluded.last_validated_at,
                last_validation_status = excluded.last_validation_status,
                last_validation_error = excluded.last_validation_error;
            """,
            (
                app_id,
                app_name,
                SOURCE,
                LANGUAGE,
                COUNTRY,
                store_info.get("title"),
                store_info.get("score"),
                store_info.get("ratings"),
                store_info.get("installs"),
                validated_at,
                "ok",
                ""
            )
        )

    except Exception as e:
        error_text = str(e)

        validation_rows.append({
            "app_name": app_name,
            "app_id": app_id,
            "validation_status": "error",
            "title_from_store": None,
            "score_from_store": None,
            "ratings_from_store": None,
            "installs_from_store": None,
            "validation_error": error_text
        })

        conn.execute(
            """
            INSERT INTO phase2_apps (
                app_id,
                app_name,
                source,
                language,
                country,
                last_validated_at,
                last_validation_status,
                last_validation_error
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(app_id) DO UPDATE SET
                app_name = excluded.app_name,
                source = excluded.source,
                language = excluded.language,
                country = excluded.country,
                last_validated_at = excluded.last_validated_at,
                last_validation_status = excluded.last_validation_status,
                last_validation_error = excluded.last_validation_error;
            """,
            (
                app_id,
                app_name,
                SOURCE,
                LANGUAGE,
                COUNTRY,
                validated_at,
                "error",
                error_text
            )
        )

conn.commit()

validation_df = pd.DataFrame(validation_rows)

print("App validation result:")
display(validation_df)

App validation result:


,app_name,app_id,validation_status,title_from_store,score_from_store,ratings_from_store,installs_from_store,validation_error
0,YouTube,com.google.android.youtube,ok,YouTube,3.861226,170926411,"10,000,000,000+",
1,TikTok,com.zhiliaoapp.musically,ok,"TikTok - Videos, Shop & LIVE",3.991081,69280732,"1,000,000,000+",
2,Spotify,com.spotify.music,ok,Spotify: Music and Podcasts,4.335805,35890307,"1,000,000,000+",
3,Instagram,com.instagram.android,ok,Instagram,4.002019,168329152,"5,000,000,000+",
4,Uber,com.ubercab,ok,Uber - Request a ride,4.743543,19073493,"1,000,000,000+",
5,DoorDash,com.dd.doordash,ok,"DoorDash: Food, Grocery, More",4.657277,6029493,"50,000,000+",
6,Duolingo,com.duolingo,ok,Duolingo: Language Lessons,4.726991,47254150,"500,000,000+",
7,Google Maps,com.google.android.apps.maps,ok,Google Maps,3.248390,19469101,"10,000,000,000+",
8,Netflix,com.netflix.mediaclient,ok,Netflix,3.870859,15170396,"1,000,000,000+",
9,Reddit,com.reddit.frontpage,ok,Reddit,4.585813,4691886,"100,000,000+",


## 10. Helper Functions for Review Processing

In [15]:
def normalize_text(value):
    if value is None:
        return None

    text = str(value)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def to_iso(value):
    if value is None:
        return None

    if isinstance(value, datetime):
        if value.tzinfo is None:
            return value.replace(tzinfo=timezone.utc).isoformat()
        return value.isoformat()

    return str(value)

def make_review_key(source, app_id, review_id, content, review_created_at, user_name):
    if review_id is not None and str(review_id).strip() != "":
        base = f"{source}|{app_id}|{review_id}"
    else:
        fallback = f"{source}|{app_id}|{content}|{review_created_at}|{user_name}"
        fallback_hash = hashlib.sha256(fallback.encode("utf-8")).hexdigest()
        base = f"{source}|{app_id}|missing_review_id|{fallback_hash}"

    return hashlib.sha256(base.encode("utf-8")).hexdigest()

def make_flag(review_key, run_id, app_id, flag_name, flag_severity, flag_value):
    flag_base = f"{review_key}|{run_id}|{flag_name}|{flag_value}"
    flag_id = hashlib.sha256(flag_base.encode("utf-8")).hexdigest()

    return {
        "flag_id": flag_id,
        "review_key": review_key,
        "run_id": run_id,
        "app_id": app_id,
        "flag_name": flag_name,
        "flag_severity": flag_severity,
        "flag_value": str(flag_value),
        "created_at": datetime.now(timezone.utc).isoformat()
    }

def get_quality_flags(review_row):
    flags = []

    review_key = review_row["review_key"]
    run_id = review_row["run_id"]
    app_id = review_row["app_id"]

    review_id = review_row.get("review_id")
    content_raw = review_row.get("content_raw")
    content_cleaned = normalize_text(content_raw)
    score = review_row.get("score")
    review_created_at = review_row.get("review_created_at")
    app_version = review_row.get("app_version")
    reply_content_raw = review_row.get("reply_content_raw")

    if review_id is None or str(review_id).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_review_id", "high", "missing"))

    if content_raw is None:
        flags.append(make_flag(review_key, run_id, app_id, "missing_content", "high", "missing"))
    elif content_cleaned == "":
        flags.append(make_flag(review_key, run_id, app_id, "empty_content", "medium", "empty"))

    if score is None:
        flags.append(make_flag(review_key, run_id, app_id, "missing_score", "high", "missing"))
    else:
        try:
            score_int = int(score)
            if score_int < 1 or score_int > 5:
                flags.append(make_flag(review_key, run_id, app_id, "invalid_score", "high", score_int))
        except Exception:
            flags.append(make_flag(review_key, run_id, app_id, "invalid_score", "high", score))

    if review_created_at is None or str(review_created_at).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_review_date", "high", "missing"))

    if app_version is None or str(app_version).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_app_version", "info", "missing"))

    if reply_content_raw is None or str(reply_content_raw).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_developer_reply", "info", "missing"))

    return flags

def raw_review_to_row(raw_review, app_name, app_id, fetched_at):
    review_id = raw_review.get("reviewId")
    content_raw = raw_review.get("content")
    review_created_at = to_iso(raw_review.get("at"))
    user_name = raw_review.get("userName")

    review_key = make_review_key(
        SOURCE,
        app_id,
        review_id,
        content_raw,
        review_created_at,
        user_name
    )

    app_version = raw_review.get("reviewCreatedVersion")

    if app_version is None:
        app_version = raw_review.get("appVersion")

    row = {
        "review_key": review_key,
        "source": SOURCE,
        "app_id": app_id,
        "app_name": app_name,
        "review_id": review_id,
        "user_name": user_name,
        "user_image": raw_review.get("userImage"),
        "content_raw": content_raw,
        "score": raw_review.get("score"),
        "thumbs_up_count": raw_review.get("thumbsUpCount"),
        "review_created_at": review_created_at,
        "reply_content_raw": raw_review.get("replyContent"),
        "replied_at": to_iso(raw_review.get("repliedAt")),
        "app_version": app_version,
        "fetched_at": fetched_at,
        "run_id": RUN_ID,
        "raw_json": json.dumps(raw_review, ensure_ascii=False, default=str)
    }

    return row

print("Review helper functions are ready.")

Review helper functions are ready.


## 11. Fetch Reviews and Insert New Records

For Day 2, most fetched reviews should already exist from Day 1.

Expected pattern:

- fetched records: 12,000
- inserted records: small number
- duplicates skipped: large number
- errors: 0

In [16]:
app_summary_rows = []
all_fetched_rows_for_export = []

for app_name, app_id in APPS.items():
    print("\n" + "=" * 90)
    print(f"Starting app: {app_name} ({app_id})")
    print("=" * 90)

    app_start_time = time.perf_counter()
    fetched_at = datetime.now(timezone.utc).isoformat()

    records_fetched = 0
    unique_reviews_in_batch = 0
    duplicate_reviews_in_batch = 0
    new_records_inserted = 0
    duplicates_skipped = 0
    inserted_quality_flags_count = 0
    error_message = ""

    raw_review_rows = []
    batch_quality_flags = []

    try:
        result, continuation_token = reviews(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=TARGET_REVIEWS_PER_APP
        )

        result = result or []
        records_fetched = len(result)

        for raw_review in result:
            row = raw_review_to_row(raw_review, app_name, app_id, fetched_at)
            raw_review_rows.append(row)
            all_fetched_rows_for_export.append(row)

        review_keys = [row["review_key"] for row in raw_review_rows]
        unique_reviews_in_batch = len(set(review_keys))
        duplicate_reviews_in_batch = records_fetched - unique_reviews_in_batch

        for row in raw_review_rows:
            flags_for_row = get_quality_flags(row)
            batch_quality_flags.extend(flags_for_row)

            cursor = conn.execute(
                """
                INSERT OR IGNORE INTO phase2_reviews_raw (
                    review_key,
                    source,
                    app_id,
                    app_name,
                    review_id,
                    user_name,
                    user_image,
                    content_raw,
                    score,
                    thumbs_up_count,
                    review_created_at,
                    reply_content_raw,
                    replied_at,
                    app_version,
                    fetched_at,
                    run_id,
                    raw_json
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
                """,
                (
                    row["review_key"],
                    row["source"],
                    row["app_id"],
                    row["app_name"],
                    row["review_id"],
                    row["user_name"],
                    row["user_image"],
                    row["content_raw"],
                    row["score"],
                    row["thumbs_up_count"],
                    row["review_created_at"],
                    row["reply_content_raw"],
                    row["replied_at"],
                    row["app_version"],
                    row["fetched_at"],
                    row["run_id"],
                    row["raw_json"]
                )
            )

            inserted_this_row = cursor.rowcount

            if inserted_this_row == 1:
                new_records_inserted += 1

                cleaned_text = normalize_text(row["content_raw"])
                content_length = len(cleaned_text) if cleaned_text is not None else 0

                if row["reply_content_raw"] is None or str(row["reply_content_raw"]).strip() == "":
                    has_developer_reply = 0
                else:
                    has_developer_reply = 1

                conn.execute(
                    """
                    INSERT OR REPLACE INTO phase2_reviews_cleaned (
                        review_key,
                        source,
                        app_id,
                        content_cleaned,
                        content_length,
                        has_developer_reply,
                        score,
                        review_created_at,
                        app_version,
                        cleaned_at,
                        run_id
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
                    """,
                    (
                        row["review_key"],
                        row["source"],
                        row["app_id"],
                        cleaned_text,
                        content_length,
                        has_developer_reply,
                        row["score"],
                        row["review_created_at"],
                        row["app_version"],
                        datetime.now(timezone.utc).isoformat(),
                        RUN_ID
                    )
                )

            for flag in flags_for_row:
                flag_cursor = conn.execute(
                    """
                    INSERT OR IGNORE INTO phase2_quality_flags (
                        flag_id,
                        review_key,
                        run_id,
                        app_id,
                        flag_name,
                        flag_severity,
                        flag_value,
                        created_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?);
                    """,
                    (
                        flag["flag_id"],
                        flag["review_key"],
                        flag["run_id"],
                        flag["app_id"],
                        flag["flag_name"],
                        flag["flag_severity"],
                        flag["flag_value"],
                        flag["created_at"]
                    )
                )

                inserted_quality_flags_count += flag_cursor.rowcount

        duplicates_skipped = records_fetched - new_records_inserted

        review_dates = [
            row["review_created_at"]
            for row in raw_review_rows
            if row["review_created_at"] is not None
        ]

        if len(review_dates) > 0:
            min_review_date = min(review_dates)
            max_review_date = max(review_dates)
        else:
            min_review_date = None
            max_review_date = None

    except Exception as e:
        min_review_date = None
        max_review_date = None
        error_message = str(e)
        print("Error:", error_message)

    app_runtime_seconds = time.perf_counter() - app_start_time

    missing_review_id_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "missing_review_id")
    missing_content_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "missing_content")
    empty_content_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "empty_content")
    missing_score_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "missing_score")
    invalid_score_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "invalid_score")
    missing_review_date_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "missing_review_date")
    missing_app_version_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "missing_app_version")
    missing_developer_reply_count = sum(1 for flag in batch_quality_flags if flag["flag_name"] == "missing_developer_reply")
    quality_flag_count = len(batch_quality_flags)

    summary_row = {
        "run_id": RUN_ID,
        "app_name": app_name,
        "app_id": app_id,
        "target_reviews": TARGET_REVIEWS_PER_APP,
        "records_fetched": records_fetched,
        "unique_reviews_in_batch": unique_reviews_in_batch,
        "duplicate_reviews_in_batch": duplicate_reviews_in_batch,
        "new_records_inserted": new_records_inserted,
        "duplicates_skipped": duplicates_skipped,
        "runtime_seconds": round(app_runtime_seconds, 2),
        "min_review_date": min_review_date,
        "max_review_date": max_review_date,
        "missing_review_id_count": missing_review_id_count,
        "missing_content_count": missing_content_count,
        "empty_content_count": empty_content_count,
        "missing_score_count": missing_score_count,
        "invalid_score_count": invalid_score_count,
        "missing_review_date_count": missing_review_date_count,
        "missing_app_version_count": missing_app_version_count,
        "missing_developer_reply_count": missing_developer_reply_count,
        "quality_flag_count": quality_flag_count,
        "quality_flags_inserted": inserted_quality_flags_count,
        "error_message": error_message
    }

    app_summary_rows.append(summary_row)

    conn.execute(
        """
        INSERT OR REPLACE INTO phase2_app_run_summary (
            run_id,
            app_name,
            app_id,
            target_reviews,
            records_fetched,
            unique_reviews_in_batch,
            duplicate_reviews_in_batch,
            new_records_inserted,
            duplicates_skipped,
            runtime_seconds,
            min_review_date,
            max_review_date,
            missing_review_id_count,
            missing_content_count,
            empty_content_count,
            missing_score_count,
            invalid_score_count,
            missing_review_date_count,
            missing_app_version_count,
            missing_developer_reply_count,
            quality_flag_count,
            error_message
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
        """,
        (
            summary_row["run_id"],
            summary_row["app_name"],
            summary_row["app_id"],
            summary_row["target_reviews"],
            summary_row["records_fetched"],
            summary_row["unique_reviews_in_batch"],
            summary_row["duplicate_reviews_in_batch"],
            summary_row["new_records_inserted"],
            summary_row["duplicates_skipped"],
            summary_row["runtime_seconds"],
            summary_row["min_review_date"],
            summary_row["max_review_date"],
            summary_row["missing_review_id_count"],
            summary_row["missing_content_count"],
            summary_row["empty_content_count"],
            summary_row["missing_score_count"],
            summary_row["invalid_score_count"],
            summary_row["missing_review_date_count"],
            summary_row["missing_app_version_count"],
            summary_row["missing_developer_reply_count"],
            summary_row["quality_flag_count"],
            summary_row["error_message"]
        )
    )

    conn.commit()

    print(f"Fetched: {records_fetched}")
    print(f"Inserted new: {new_records_inserted}")
    print(f"Duplicates skipped: {duplicates_skipped}")
    print(f"Quality flags in fetched batch: {quality_flag_count}")
    print(f"Quality flags inserted for this run: {inserted_quality_flags_count}")
    print(f"Runtime seconds: {app_runtime_seconds:.2f}")

    time.sleep(1)

app_summary_df = pd.DataFrame(app_summary_rows)

print("\nApp-level run summary:")
display(app_summary_df)


Starting app: YouTube (com.google.android.youtube)
Fetched: 1200
Inserted new: 33
Duplicates skipped: 1167
Quality flags in fetched batch: 1221
Quality flags inserted for this run: 1221
Runtime seconds: 1.02

Starting app: TikTok (com.zhiliaoapp.musically)
Fetched: 1200
Inserted new: 7
Duplicates skipped: 1193
Quality flags in fetched batch: 630
Quality flags inserted for this run: 630
Runtime seconds: 0.78

Starting app: Spotify (com.spotify.music)
Fetched: 1200
Inserted new: 15
Duplicates skipped: 1185
Quality flags in fetched batch: 1247
Quality flags inserted for this run: 1247
Runtime seconds: 0.71

Starting app: Instagram (com.instagram.android)
Fetched: 1200
Inserted new: 52
Duplicates skipped: 1148
Quality flags in fetched batch: 1584
Quality flags inserted for this run: 1584
Runtime seconds: 0.98

Starting app: Uber (com.ubercab)
Fetched: 1200
Inserted new: 13
Duplicates skipped: 1187
Quality flags in fetched batch: 1367
Quality flags inserted for this run: 1367
Runtime secon

,run_id,app_name,app_id,target_reviews,records_fetched,unique_reviews_in_batch,duplicate_reviews_in_batch,new_records_inserted,duplicates_skipped,runtime_seconds,min_review_date,max_review_date,missing_review_id_count,missing_content_count,empty_content_count,missing_score_count,invalid_score_count,missing_review_date_count,missing_app_version_count,missing_developer_reply_count,quality_flag_count,quality_flags_inserted,error_message
0,phase2_day2_daily_followup_20260708_041135,YouTube,com.google.android.youtube,1200,1200,1200,0,33,1167,1.02,2026-07-06T09:48:19+00:00,2026-07-07T04:14:03+00:00,0,0,0,0,0,0,21,1200,1221,1221,
1,phase2_day2_daily_followup_20260708_041135,TikTok,com.zhiliaoapp.musically,1200,1200,1200,0,7,1193,0.78,2026-07-05T03:35:23+00:00,2026-07-07T04:14:30+00:00,0,0,0,0,0,0,448,182,630,630,
2,phase2_day2_daily_followup_20260708_041135,Spotify,com.spotify.music,1200,1200,1200,0,15,1185,0.71,2026-07-05T13:12:10+00:00,2026-07-07T04:15:01+00:00,0,0,0,0,0,0,190,1057,1247,1247,
3,phase2_day2_daily_followup_20260708_041135,Instagram,com.instagram.android,1200,1200,1200,0,52,1148,0.98,2026-07-06T14:04:28+00:00,2026-07-07T04:16:51+00:00,0,0,0,0,0,0,384,1200,1584,1584,
4,phase2_day2_daily_followup_20260708_041135,Uber,com.ubercab,1200,1200,1200,0,13,1187,0.63,2026-07-04T04:37:57+00:00,2026-07-07T04:16:54+00:00,0,0,0,0,0,0,170,1197,1367,1367,
5,phase2_day2_daily_followup_20260708_041135,DoorDash,com.dd.doordash,1200,1200,1200,0,0,1200,0.73,2026-06-28T23:28:16+00:00,2026-07-07T03:41:39+00:00,0,0,0,0,0,0,126,1200,1326,1326,
6,phase2_day2_daily_followup_20260708_041135,Duolingo,com.duolingo,1200,1200,1200,0,28,1172,0.76,2026-07-06T05:42:10+00:00,2026-07-07T04:15:00+00:00,0,0,0,0,0,0,76,1200,1276,1276,
7,phase2_day2_daily_followup_20260708_041135,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,3,1197,0.87,2026-06-30T03:12:23+00:00,2026-07-07T04:05:23+00:00,0,0,0,0,0,0,30,935,965,965,
8,phase2_day2_daily_followup_20260708_041135,Netflix,com.netflix.mediaclient,1200,1200,1200,0,2,1198,0.80,2026-06-27T05:51:36+00:00,2026-07-07T04:10:52+00:00,0,0,0,0,0,0,378,1200,1578,1578,
9,phase2_day2_daily_followup_20260708_041135,Reddit,com.reddit.frontpage,1200,1200,1200,0,3,1197,0.60,2026-06-27T05:16:47+00:00,2026-07-07T04:16:32+00:00,0,0,0,0,0,0,244,1200,1444,1444,


## 12. Save Fetched Review Export

In [17]:
fetched_export_df = pd.DataFrame(all_fetched_rows_for_export)

fetched_export_path = OUTPUT_DIR / f"{RUN_LABEL}_fetched_reviews_{run_timestamp}.csv"

if len(fetched_export_df) > 0:
    export_cols = [
        "run_id",
        "source",
        "app_name",
        "app_id",
        "review_id",
        "score",
        "review_created_at",
        "app_version",
        "content_raw",
        "reply_content_raw",
        "fetched_at"
    ]

    export_cols = [
        col for col in export_cols
        if col in fetched_export_df.columns
    ]

    fetched_export_df[export_cols].to_csv(fetched_export_path, index=False)
    print("Fetched review export saved to:", fetched_export_path)
else:
    print("No fetched reviews to export.")

Fetched review export saved to: outputs/phase2_day2_daily_followup_fetched_reviews_20260708_041135.csv


## 13. Build Final Day 2 Run Summary

In [18]:
run_finished_at = datetime.now(timezone.utc)
run_finished_at_text = run_finished_at.isoformat()
runtime_seconds = (run_finished_at - run_started_at).total_seconds()

if len(app_summary_df) > 0:
    records_fetched_total = int(app_summary_df["records_fetched"].sum())
    new_records_inserted_total = int(app_summary_df["new_records_inserted"].sum())
    duplicates_skipped_total = int(app_summary_df["duplicates_skipped"].sum())
    errors_total = int((app_summary_df["error_message"].fillna("") != "").sum())
    quality_flag_total = int(app_summary_df["quality_flag_count"].sum())
    quality_flags_inserted = int(app_summary_df["quality_flags_inserted"].sum())
else:
    records_fetched_total = 0
    new_records_inserted_total = 0
    duplicates_skipped_total = 0
    errors_total = 0
    quality_flag_total = 0
    quality_flags_inserted = 0

apps_failed = ", ".join(
    app_summary_df.loc[
        app_summary_df["error_message"].fillna("") != "",
        "app_name"
    ].tolist()
)

db_size_after_mb = get_db_size_mb(DB_PATH)
db_size_growth_mb = db_size_after_mb - db_size_before_mb

review_rows_after = count_rows(conn, "phase2_reviews_raw")
review_rows_growth = review_rows_after - review_rows_before

if errors_total > 0:
    run_status = "completed_with_errors"
else:
    run_status = "completed"

conn.execute(
    """
    UPDATE phase2_ingestion_runs
    SET
        run_finished_at = ?,
        runtime_seconds = ?,
        status = ?,
        records_fetched_total = ?,
        new_records_inserted_total = ?,
        duplicates_skipped_total = ?,
        errors_total = ?,
        apps_failed = ?,
        quality_flag_total = ?,
        quality_flags_inserted = ?,
        db_size_after_mb = ?,
        db_size_growth_mb = ?,
        review_rows_after = ?,
        review_rows_growth = ?
    WHERE run_id = ?;
    """,
    (
        run_finished_at_text,
        runtime_seconds,
        run_status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        apps_failed,
        quality_flag_total,
        quality_flags_inserted,
        db_size_after_mb,
        db_size_growth_mb,
        review_rows_after,
        review_rows_growth,
        RUN_ID
    )
)

conn.commit()

run_summary_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_label": RUN_LABEL,
    "phase": PHASE,
    "frequency_label": FREQUENCY_LABEL,
    "source": SOURCE,
    "language": LANGUAGE,
    "country": COUNTRY,
    "target_reviews_per_app": TARGET_REVIEWS_PER_APP,
    "app_count": len(APPS),
    "apps_included": ", ".join(APPS.keys()),
    "run_started_at": run_started_at_text,
    "run_finished_at": run_finished_at_text,
    "runtime_seconds": round(runtime_seconds, 2),
    "status": run_status,
    "records_fetched_total": records_fetched_total,
    "new_records_inserted_total": new_records_inserted_total,
    "duplicates_skipped_total": duplicates_skipped_total,
    "errors_total": errors_total,
    "apps_failed": apps_failed,
    "quality_flag_total": quality_flag_total,
    "quality_flags_inserted": quality_flags_inserted,
    "db_size_before_mb": round(db_size_before_mb, 4),
    "db_size_after_mb": round(db_size_after_mb, 4),
    "db_size_growth_mb": round(db_size_growth_mb, 4),
    "review_rows_before": review_rows_before,
    "review_rows_after": review_rows_after,
    "review_rows_growth": review_rows_growth
}])

print("Run summary:")
display(run_summary_df)

Run summary:


,run_id,run_label,phase,frequency_label,source,language,country,target_reviews_per_app,app_count,apps_included,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,apps_failed,quality_flag_total,quality_flags_inserted,db_size_before_mb,db_size_after_mb,db_size_growth_mb,review_rows_before,review_rows_after,review_rows_growth
0,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,phase2,daily_followup,google_play,en,us,1200,10,"YouTube, TikTok, Spotify, Instagram, Uber, DoorDash, Duolingo, Google Maps, Netflix, Reddit",2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,75.8,completed,12000,156,11844,0,,12638,12638,25.4922,30.1875,4.6953,12000,12156,156


## 14. Save Day 2 Summary Files

In [19]:
run_summary_path = RUN_SUMMARY_DIR / f"phase2_day2_run_summary_{run_timestamp}.csv"
app_summary_path = RUN_SUMMARY_DIR / f"phase2_day2_app_level_summary_{run_timestamp}.csv"
history_path = RUN_SUMMARY_DIR / "phase2_run_summary_history.csv"

run_summary_df.to_csv(run_summary_path, index=False)
app_summary_df.to_csv(app_summary_path, index=False)

history_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        apps_failed,
        quality_flag_total,
        quality_flags_inserted,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        review_rows_before,
        review_rows_after,
        review_rows_growth
    FROM phase2_ingestion_runs
    WHERE run_label IN ('phase2_day1_controlled_scale', 'phase2_day2_daily_followup')
    ORDER BY run_started_at;
    """,
    conn
)

history_df.to_csv(history_path, index=False)

print("Run summary saved to:", run_summary_path)
print("App-level summary saved to:", app_summary_path)
print("Run summary history saved to:", history_path)

print("Phase 2 run summary history:")
display(history_df)

Run summary saved to: outputs/run_summaries/phase2_day2_run_summary_20260708_041135.csv
App-level summary saved to: outputs/run_summaries/phase2_day2_app_level_summary_20260708_041135.csv
Run summary history saved to: outputs/run_summaries/phase2_run_summary_history.csv
Phase 2 run summary history:


,run_id,run_label,phase,frequency_label,source,language,country,target_reviews_per_app,app_count,apps_included,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,apps_failed,quality_flag_total,quality_flags_inserted,db_size_before_mb,db_size_after_mb,db_size_growth_mb,review_rows_before,review_rows_after,review_rows_growth
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,phase2,once_daily_baseline,google_play,en,us,1200,10,"YouTube, TikTok, Spotify, Instagram, Uber, DoorDash, Duolingo, Google Maps, Netflix, Reddit",2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,23.271117,completed,12000,12000,0,0,,12633,12633,1.832031,25.492188,23.660156,0,12000,12000
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,phase2,daily_followup,google_play,en,us,1200,10,"YouTube, TikTok, Spotify, Instagram, Uber, DoorDash, Duolingo, Google Maps, Netflix, Reddit",2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,75.799585,completed,12000,156,11844,0,,12638,12638,25.492188,30.187500,4.695312,12000,12156,156


## 15. Quality Flag Summary

In [20]:
quality_flag_summary_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        flag_name,
        flag_severity,
        COUNT(*) AS flag_count
    FROM phase2_quality_flags
    WHERE run_id = ?
    GROUP BY run_id, flag_name, flag_severity
    ORDER BY flag_severity, flag_name;
    """,
    conn,
    params=[RUN_ID]
)

quality_flag_summary_path = QUALITY_DIR / f"phase2_day2_quality_flag_summary_{run_timestamp}.csv"
quality_flag_summary_df.to_csv(quality_flag_summary_path, index=False)

print("Quality flag summary:")
display(quality_flag_summary_df)

print("Quality flag summary saved to:", quality_flag_summary_path)

Quality flag summary:


,run_id,flag_name,flag_severity,flag_count
0,phase2_day2_daily_followup_20260708_041135,missing_app_version,info,2067
1,phase2_day2_daily_followup_20260708_041135,missing_developer_reply,info,10571


Quality flag summary saved to: outputs/quality/phase2_day2_quality_flag_summary_20260708_041135.csv


## 16. Relationship Checks

In [21]:
raw_without_app = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_reviews_raw r
    LEFT JOIN phase2_apps a
        ON r.app_id = a.app_id
    WHERE a.app_id IS NULL;
    """,
    conn
)["issue_count"].iloc[0]

cleaned_without_raw = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_reviews_cleaned c
    LEFT JOIN phase2_reviews_raw r
        ON c.review_key = r.review_key
    WHERE r.review_key IS NULL;
    """,
    conn
)["issue_count"].iloc[0]

flags_without_raw = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_quality_flags q
    LEFT JOIN phase2_reviews_raw r
        ON q.review_key = r.review_key
    WHERE r.review_key IS NULL;
    """,
    conn
)["issue_count"].iloc[0]

flags_without_run = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_quality_flags q
    LEFT JOIN phase2_ingestion_runs ir
        ON q.run_id = ir.run_id
    WHERE q.run_id = ?
      AND ir.run_id IS NULL;
    """,
    conn,
    params=[RUN_ID]
)["issue_count"].iloc[0]

duplicate_review_keys = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM (
        SELECT
            source,
            app_id,
            review_id,
            COUNT(*) AS cnt
        FROM phase2_reviews_raw
        WHERE review_id IS NOT NULL
          AND TRIM(review_id) != ''
        GROUP BY source, app_id, review_id
        HAVING COUNT(*) > 1
    );
    """,
    conn
)["issue_count"].iloc[0]

run_rows_without_run = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_reviews_raw r
    LEFT JOIN phase2_ingestion_runs ir
        ON r.run_id = ir.run_id
    WHERE r.run_id = ?
      AND ir.run_id IS NULL;
    """,
    conn,
    params=[RUN_ID]
)["issue_count"].iloc[0]

relationship_checks = [
    {
        "check_name": "raw_reviews_have_app_record",
        "issue_count": int(raw_without_app),
        "status": "pass" if raw_without_app == 0 else "review_needed"
    },
    {
        "check_name": "cleaned_reviews_link_to_raw_reviews",
        "issue_count": int(cleaned_without_raw),
        "status": "pass" if cleaned_without_raw == 0 else "review_needed"
    },
    {
        "check_name": "quality_flags_link_to_raw_reviews",
        "issue_count": int(flags_without_raw),
        "status": "pass" if flags_without_raw == 0 else "review_needed"
    },
    {
        "check_name": "quality_flags_link_to_ingestion_run",
        "issue_count": int(flags_without_run),
        "status": "pass" if flags_without_run == 0 else "review_needed"
    },
    {
        "check_name": "no_duplicate_source_app_review_id",
        "issue_count": int(duplicate_review_keys),
        "status": "pass" if duplicate_review_keys == 0 else "review_needed"
    },
    {
        "check_name": "run_reviews_link_to_ingestion_run",
        "issue_count": int(run_rows_without_run),
        "status": "pass" if run_rows_without_run == 0 else "review_needed"
    }
]

relationship_checks_df = pd.DataFrame(relationship_checks)

relationship_check_path = RUN_SUMMARY_DIR / f"phase2_day2_relationship_checks_{run_timestamp}.csv"
relationship_checks_df.to_csv(relationship_check_path, index=False)

print("Database relationship checks:")
display(relationship_checks_df)

print("Relationship checks saved to:", relationship_check_path)

Database relationship checks:


,check_name,issue_count,status
0,raw_reviews_have_app_record,0,pass
1,cleaned_reviews_link_to_raw_reviews,0,pass
2,quality_flags_link_to_raw_reviews,0,pass
3,quality_flags_link_to_ingestion_run,0,pass
4,no_duplicate_source_app_review_id,0,pass
5,run_reviews_link_to_ingestion_run,0,pass


Relationship checks saved to: outputs/run_summaries/phase2_day2_relationship_checks_20260708_041135.csv


## 17. Database Row Count Growth

In [22]:
tables_after_df = get_tables(conn)

row_counts_after = []

for table_name in tables_after_df["name"]:
    row_counts_after.append({
        "table_name": table_name,
        "row_count_after": count_rows(conn, table_name)
    })

row_counts_after_df = pd.DataFrame(row_counts_after)

row_count_comparison_df = row_counts_before_df.merge(
    row_counts_after_df,
    on="table_name",
    how="outer"
).fillna(0)

row_count_comparison_df["row_count_before"] = row_count_comparison_df["row_count_before"].astype(int)
row_count_comparison_df["row_count_after"] = row_count_comparison_df["row_count_after"].astype(int)

row_count_comparison_df["row_growth"] = (
    row_count_comparison_df["row_count_after"] - row_count_comparison_df["row_count_before"]
)

row_count_comparison_path = RUN_SUMMARY_DIR / f"phase2_day2_database_row_growth_{run_timestamp}.csv"
row_count_comparison_df.to_csv(row_count_comparison_path, index=False)

print("Database row count comparison:")
display(row_count_comparison_df)

print("Database row growth file saved to:", row_count_comparison_path)

Database row count comparison:


,table_name,row_count_before,row_count_after,row_growth
0,app_sources,3,3,0
1,ingestion_run_targets,12,12,0
2,ingestion_runs,4,4,0
3,phase2_app_run_summary,20,20,0
4,phase2_apps,10,10,0
5,phase2_ingestion_runs,2,2,0
6,phase2_quality_flags,12633,25271,12638
7,phase2_reviews_cleaned,12000,12156,156
8,phase2_reviews_raw,12000,12156,156
9,review_quality_flags,1200,1200,0


Database row growth file saved to: outputs/run_summaries/phase2_day2_database_row_growth_20260708_041135.csv


## 18. Day 1 vs Day 2 Comparison

In [23]:
day1_day2_comparison_df = history_df[
    history_df["run_label"].isin([
        "phase2_day1_controlled_scale",
        "phase2_day2_daily_followup"
    ])
].copy()

comparison_cols = [
    "run_label",
    "frequency_label",
    "status",
    "records_fetched_total",
    "new_records_inserted_total",
    "duplicates_skipped_total",
    "errors_total",
    "quality_flag_total",
    "db_size_before_mb",
    "db_size_after_mb",
    "db_size_growth_mb",
    "review_rows_before",
    "review_rows_after",
    "review_rows_growth",
    "runtime_seconds"
]

comparison_cols = [
    col for col in comparison_cols
    if col in day1_day2_comparison_df.columns
]

day1_day2_comparison_df = day1_day2_comparison_df[comparison_cols]

day1_day2_comparison_path = RUN_SUMMARY_DIR / f"phase2_day1_day2_comparison_{run_timestamp}.csv"
day1_day2_comparison_df.to_csv(day1_day2_comparison_path, index=False)

print("Day 1 vs Day 2 comparison:")
display(day1_day2_comparison_df)

print("Day 1 vs Day 2 comparison saved to:", day1_day2_comparison_path)

Day 1 vs Day 2 comparison:


,run_label,frequency_label,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,quality_flag_total,db_size_before_mb,db_size_after_mb,db_size_growth_mb,review_rows_before,review_rows_after,review_rows_growth,runtime_seconds
0,phase2_day1_controlled_scale,once_daily_baseline,completed,12000,12000,0,0,12633,1.832031,25.492188,23.660156,0,12000,12000,23.271117
1,phase2_day2_daily_followup,daily_followup,completed,12000,156,11844,0,12638,25.492188,30.187500,4.695312,12000,12156,156,75.799585


Day 1 vs Day 2 comparison saved to: outputs/run_summaries/phase2_day1_day2_comparison_20260708_041135.csv


## 19. Final Clean Validation Before Export

This confirms the final database has only clean Day 1 and Day 2 run records.

In [24]:
final_run_check_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        review_rows_before,
        review_rows_after
    FROM phase2_ingestion_runs
    ORDER BY run_started_at;
    """,
    conn
)

final_failed_runs_df = final_run_check_df[
    final_run_check_df["status"] != "completed"
]

final_day1_count = int((final_run_check_df["run_label"] == "phase2_day1_controlled_scale").sum())
final_day2_count = int((final_run_check_df["run_label"] == "phase2_day2_daily_followup").sum())

print("Final run check:")
display(final_run_check_df)

print("Final failed runs:")
display(final_failed_runs_df)

if len(final_run_check_df) != 2:
    raise ValueError("Final database should contain exactly 2 Phase 2 runs: Day 1 and Day 2.")

if final_day1_count != 1:
    raise ValueError("Final database should contain exactly 1 Day 1 run.")

if final_day2_count != 1:
    raise ValueError("Final database should contain exactly 1 Day 2 run.")

if len(final_failed_runs_df) > 0:
    raise ValueError("Final database contains failed runs. Do not upload this version.")

if int(final_run_check_df.loc[final_run_check_df["run_label"] == "phase2_day1_controlled_scale", "new_records_inserted_total"].iloc[0]) != 12000:
    raise ValueError("Final Day 1 inserted count is not 12,000.")

if int(final_run_check_df.loc[final_run_check_df["run_label"] == "phase2_day2_daily_followup", "errors_total"].iloc[0]) != 0:
    raise ValueError("Final Day 2 errors_total is not 0.")

print("Final clean validation passed.")

Final run check:


,run_id,run_label,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,review_rows_before,review_rows_after
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,completed,12000,12000,0,0,0,12000
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,completed,12000,156,11844,0,12000,12156


Final failed runs:


,run_id,run_label,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,review_rows_before,review_rows_after


Final clean validation passed.


## 20. Write Findings Report

In [25]:
def df_to_markdown_table(df):
    if df is None or len(df) == 0:
        return "_No rows._"

    safe_df = df.copy()
    safe_df = safe_df.fillna("")

    headers = list(safe_df.columns)
    lines = []

    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for _, row in safe_df.iterrows():
        values = [
            str(row[col]).replace("\n", " ").replace("|", "/")
            for col in headers
        ]
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)

short_app_summary_cols = [
    "app_name",
    "records_fetched",
    "new_records_inserted",
    "duplicates_skipped",
    "runtime_seconds",
    "quality_flag_count",
    "error_message"
]

short_app_summary_cols = [
    col for col in short_app_summary_cols
    if col in app_summary_df.columns
]

report_text = f"""
# Google Play Controlled Scale Ingestion — Phase 2 Day 2 Findings

## 1. Purpose

This run continues Phase 2 of the Google Play review ingestion pipeline.

Phase 2 Day 1 created the controlled baseline database with 12,000 reviews across 10 apps. Phase 2 Day 2 repeats the same ingestion scope using the clean Day 1 database to test duplicate handling, new review capture, run stability, quality flags, and database growth.

## 2. Test Scope

- Source: {SOURCE}
- Language / country: {LANGUAGE} / {COUNTRY}
- Apps tested: {len(APPS)}
- Target reviews per app: {TARGET_REVIEWS_PER_APP}
- Database used: `{DB_PATH}`
- Run ID: `{RUN_ID}`
- Frequency label: `{FREQUENCY_LABEL}`
- Duplicate rule: `source + app_id + review_id`

## 3. Day 2 Run Summary

{df_to_markdown_table(run_summary_df)}

## 4. Day 1 vs Day 2 Comparison

{df_to_markdown_table(day1_day2_comparison_df)}

## 5. App-Level Summary

{df_to_markdown_table(app_summary_df[short_app_summary_cols])}

## 6. Quality Flag Summary

{df_to_markdown_table(quality_flag_summary_df)}

## 7. Database Relationship Checks

{df_to_markdown_table(relationship_checks_df)}

## 8. Main Notes

- The run started from the clean Phase 2 Day 1 database.
- The same 10 apps and same target review count were used as Day 1.
- Day 2 duplicate counts are meaningful because the database already contained Day 1 records.
- The run summary captures fetched records, inserted records, duplicates skipped, errors, quality flags, database size growth, and row growth.
- Relationship checks passed after the repeated ingestion run.
- No failed Day 1 or old Day 2 runs are included in the final database history.

## 9. Next Step

The next run can use the same database and app list to test another collection time, such as a same-day evening run or a Day 3 daily follow-up run.
"""

report_path = REPORT_DIR / f"phase2_day2_daily_followup_findings_{run_timestamp}.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text.strip())

print("Findings report saved to:", report_path)

Findings report saved to: reports/phase2_day2_daily_followup_findings_20260708_041135.md


## 21. Compress Updated Database for GitHub

GitHub browser upload can fail for large `.sqlite` files.

So the updated database is stored as:

`database/google_play_reviews.sqlite.zip`

The raw `.sqlite` file is not included in the final GitHub upload package.

In [26]:
db_file = Path(DB_PATH)
db_zip_path = Path("database") / "google_play_reviews.sqlite.zip"

if not db_file.exists():
    raise FileNotFoundError("Database file not found.")

with zipfile.ZipFile(db_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(db_file, arcname="google_play_reviews.sqlite")

db_file_size_mb = os.path.getsize(db_file) / (1024 * 1024)
db_zip_size_mb = os.path.getsize(db_zip_path) / (1024 * 1024)

print(f"Raw database size: {db_file_size_mb:.4f} MB")
print(f"Compressed database zip size: {db_zip_size_mb:.4f} MB")
print("Compressed database saved to:", db_zip_path)

Raw database size: 30.1875 MB
Compressed database zip size: 7.4195 MB
Compressed database saved to: database/google_play_reviews.sqlite.zip


## 22. Prepare GitHub Upload Package

This package includes only clean Day 2 outputs and the compressed updated database.

It does not include the raw `.sqlite` file or backup files.

In [27]:
repo_dir = Path("/content/app-review-source-validation")

files_to_include = [
    repo_dir / "database" / "google_play_reviews.sqlite.zip",
    repo_dir / "outputs" / "run_summaries" / f"phase2_day2_run_summary_{run_timestamp}.csv",
    repo_dir / "outputs" / "run_summaries" / f"phase2_day2_app_level_summary_{run_timestamp}.csv",
    repo_dir / "outputs" / "run_summaries" / f"phase2_day2_relationship_checks_{run_timestamp}.csv",
    repo_dir / "outputs" / "run_summaries" / f"phase2_day2_database_row_growth_{run_timestamp}.csv",
    repo_dir / "outputs" / "run_summaries" / f"phase2_day1_day2_comparison_{run_timestamp}.csv",
    repo_dir / "outputs" / "run_summaries" / "phase2_run_summary_history.csv",
    repo_dir / "outputs" / "quality" / f"phase2_day2_quality_flag_summary_{run_timestamp}.csv",
    repo_dir / "reports" / f"phase2_day2_daily_followup_findings_{run_timestamp}.md",
]

existing_files_to_include = []

for f in files_to_include:
    if f.exists():
        existing_files_to_include.append(f)
    else:
        print("Missing file:", f.relative_to(repo_dir))

if len(existing_files_to_include) == 0:
    raise FileNotFoundError("No output files found for packaging.")

print("Files included in GitHub upload package:")
for f in existing_files_to_include:
    print("-", f.relative_to(repo_dir))

zip_path = Path(f"/content/phase2_day2_github_upload_files_{run_timestamp}.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in existing_files_to_include:
        arcname = f.relative_to(repo_dir)
        zipf.write(f, arcname)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print("\nGitHub upload package created:")
print(zip_path)
print(f"Package size: {zip_size_mb:.4f} MB")

files.download(str(zip_path))

Files included in GitHub upload package:
- database/google_play_reviews.sqlite.zip
- outputs/run_summaries/phase2_day2_run_summary_20260708_041135.csv
- outputs/run_summaries/phase2_day2_app_level_summary_20260708_041135.csv
- outputs/run_summaries/phase2_day2_relationship_checks_20260708_041135.csv
- outputs/run_summaries/phase2_day2_database_row_growth_20260708_041135.csv
- outputs/run_summaries/phase2_day1_day2_comparison_20260708_041135.csv
- outputs/run_summaries/phase2_run_summary_history.csv
- outputs/quality/phase2_day2_quality_flag_summary_20260708_041135.csv
- reports/phase2_day2_daily_followup_findings_20260708_041135.md

GitHub upload package created:
/content/phase2_day2_github_upload_files_20260708_041135.zip
Package size: 7.4254 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 23. Git Status Check

In [28]:
!git status --short

 M database/google_play_reviews.sqlite
 M database/google_play_reviews.sqlite.zip
?? database/backups/
?? google_play_reviews.sqlite.zip
?? outputs/quality/
?? outputs/run_summaries/
?? reports/phase2_day2_daily_followup_findings_20260708_041135.md
?? uploaded_clean_day1_database.sqlite


In [29]:
conn.close()
print("Database connection closed. Phase 2 Day 2 notebook completed.")

Database connection closed. Phase 2 Day 2 notebook completed.
